In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import sys
from datetime import datetime

# Project folders
PROJECT = Path.cwd()

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
REPORTS = PROJECT / "reports"
DELIVERABLES = PROJECT / "deliverables"

for folder in [RAW, PROCESSED, REPORTS, DELIVERABLES]:
    folder.mkdir(parents=True, exist_ok=True)

print("Python version:", sys.version)
print("Project folder:", PROJECT)
print("Raw data folder:", RAW)
print("Processed data folder:", PROCESSED)
print("Reports folder:", REPORTS)

In [ ]:
%pip install pandas numpy openpyxl matplotlib seaborn scikit-learn

In [ ]:
files = sorted(RAW.glob("*"))

for file in files:
    if file.is_file():
        print(f"{file.name:45} {file.stat().st_size / 1024**2:.2f} MB")

In [ ]:
excel_files = list(RAW.glob("*.xlsx")) + list(RAW.glob("*.xlsm"))

for file in excel_files:
    print(f"\n{'=' * 70}")
    print("WORKBOOK:", file.name)

    workbook = pd.ExcelFile(file)

    for sheet in workbook.sheet_names:
        preview = pd.read_excel(
            file,
            sheet_name=sheet,
            nrows=5
        )

        print(f"\nSheet: {sheet}")
        print("Columns:", list(preview.columns))
        print("Preview shape:", preview.shape)
        display(preview.head(3))

In [5]:
from openpyxl import load_workbook

inventory = []

for file in excel_files:
    wb = load_workbook(
        file,
        read_only=True,
        data_only=True
    )

    for ws in wb.worksheets:
        inventory.append({
            "workbook": file.name,
            "sheet": ws.title,
            "rows": ws.max_row,
            "columns": ws.max_column
        })

    wb.close()

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

inventory_df.to_csv(
    REPORTS / "workbook_inventory.csv",
    index=False
)

,workbook,sheet,rows,columns
0,01 Introduction.xlsx,Nomenclature,63,6
1,01 Introduction.xlsx,Uncertainty Values,101,5
2,02 Field Production.xlsx,Plots,1,1
3,02 Field Production.xlsx,Field Production,42,6
4,02 Field Production.xlsx,Gas production cumulative,11099,102
...,...,...,...,...
115,09 Template Deliverable.xlsx,Case 6,81,4
116,09 Template Deliverable.xlsx,Case 7,81,4
117,09 Template Deliverable.xlsx,Case 8,81,4
118,09 Template Deliverable.xlsx,Case 9,81,4


In [6]:
simulation_file = next(
    (p for p in RAW.glob("*.xlsx")
     if "train" in p.name.lower()),
    None
)

if simulation_file is None:
    print("Select your actual simulation workbook.")
else:
    print(simulation_file)

c:\Users\iffah\Downloads\dataworks\data\raw\03 Train Cases 1.xlsx


In [12]:
import pandas as pd

# Store information about every worksheet
workbook_report = []

for file in excel_files:
    print(f"Inspecting: {file.name}")

    try:
        workbook = pd.ExcelFile(file)

        for sheet in workbook.sheet_names:
            try:
                df = pd.read_excel(
                    file,
                    sheet_name=sheet
                )

                numeric_df = df.select_dtypes(include="number")

                placeholder_count = int(
                    numeric_df.eq(-999.25).sum().sum()
                )

                workbook_report.append({
                    "workbook": file.name,
                    "sheet": sheet,
                    "rows": len(df),
                    "columns": len(df.columns),
                    "missing_values": int(
                        df.isna().sum().sum()
                    ),
                    "duplicate_rows": int(
                        df.duplicated().sum()
                    ),
                    "placeholder_-999.25": placeholder_count
                })

            except Exception as e:
                print(f"  Could not read sheet '{sheet}': {e}")

    except Exception as e:
        print(f"  Could not open workbook: {e}")

# Display the inspection results
workbook_report_df = pd.DataFrame(workbook_report)

display(workbook_report_df)

Inspecting: 01 Introduction.xlsx
Inspecting: 02 Field Production.xlsx
Inspecting: 03 Train Cases 1.xlsx
Inspecting: 04 Train Cases 2.xlsx
Inspecting: 05 Train Cases 3.xlsx
Inspecting: 06 Train Cases 4.xlsx
Inspecting: 07  Validation Cases.xlsx
Inspecting: 08 Test Cases.xlsx
Inspecting: 09 Template Deliverable.xlsx


,workbook,sheet,rows,columns,missing_values,duplicate_rows,placeholder_-999.25
0,01 Introduction.xlsx,Nomenclature,32,4,40,0,0
1,01 Introduction.xlsx,Uncertainty Values,100,5,0,0,0
2,02 Field Production.xlsx,Plots,0,0,0,0,0
3,02 Field Production.xlsx,Field Production,41,6,0,0,0
4,02 Field Production.xlsx,Gas production cumulative,11098,102,1103434,0,0
...,...,...,...,...,...,...,...
115,09 Template Deliverable.xlsx,Case 6,80,4,240,0,0
116,09 Template Deliverable.xlsx,Case 7,80,4,240,0,0
117,09 Template Deliverable.xlsx,Case 8,80,4,240,0,0
118,09 Template Deliverable.xlsx,Case 9,80,4,240,0,0


In [14]:
# Display all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(workbook_report_df)

,workbook,sheet,rows,columns,missing_values,duplicate_rows,placeholder_-999.25
0,01 Introduction.xlsx,Nomenclature,32,4,40,0,0
1,01 Introduction.xlsx,Uncertainty Values,100,5,0,0,0
2,02 Field Production.xlsx,Plots,0,0,0,0,0
3,02 Field Production.xlsx,Field Production,41,6,0,0,0
4,02 Field Production.xlsx,Gas production cumulative,11098,102,1103434,0,0
5,02 Field Production.xlsx,Gas production rate,11098,102,1103434,0,0
6,02 Field Production.xlsx,Oil production cumulative,11098,102,1103434,0,0
7,02 Field Production.xlsx,Oil production rate,11098,102,1103434,0,0
8,02 Field Production.xlsx,Water production cumulative,11098,102,1103434,0,0
9,02 Field Production.xlsx,Water production rate,11098,102,1103434,0,0


In [13]:
# Define where to save the report
report_path = REPORTS / "workbook_inspection_report.csv"

# Save the report as a CSV file
workbook_report_df.to_csv(
    report_path,
    index=False
)

print("Inspection report saved successfully!")
print(report_path)

Inspection report saved successfully!
c:\Users\iffah\Downloads\dataworks\reports\workbook_inspection_report.csv


In [17]:
from pathlib import Path
import pandas as pd

# Locate the raw workbooks
RAW = Path.cwd() / "data" / "raw"

files_to_inspect = {
    "01 Introduction.xlsx": [
        "Nomenclature",
        "Uncertainty Values"
    ],
    "02 Field Production.xlsx": [
        "Field Production",
        "Gas production cumulative",
        "Oil production rate",
        "Oil production cumulative"
    ],
    "03 Train Cases 1.xlsx": ["Case 1"],
    "04 Train Cases 2.xlsx": ["Case 21"],
    "05 Train Cases 3.xlsx": ["Case 41"],
    "06 Train Cases 4.xlsx": ["Case 61"],
    "07 Validation Cases.xlsx": ["Case 71"],
    "08 Test Cases.xlsx": ["Case 86"],
    "09 Template Deliverable.xlsx": ["Case 1"]
}

for filename, sheets in files_to_inspect.items():
    filepath = RAW / filename

    print("\n" + "=" * 70)
    print("WORKBOOK:", filename)

    for sheet in sheets:
        df = pd.read_excel(
            filepath,
            sheet_name=sheet,
            nrows=5
        )

        print(f"\nSHEET: {sheet}")
        print("Columns:", len(df.columns))
        print("Column names:", df.columns.tolist())

        display(df.head(3))


WORKBOOK: 01 Introduction.xlsx

SHEET: Nomenclature
Columns: 2
Column names: ['Grid', 'I Index']


,Grid,I Index
0,NaN,J Index
1,NaN,K Index
2,NaN,X Coordinate



SHEET: Uncertainty Values
Columns: 5
Column names: ['Item', 'Fault Transmissibility', 'Porosity Multiplier', 'Permeability Multiplier', 'Aquifer Pore Volume']


,Item,Fault Transmissibility,Porosity Multiplier,Permeability Multiplier,Aquifer Pore Volume
0,Case 1,0.052351,1.082307,3.047346,68.352306
1,Case 2,0.089456,1.229391,6.354067,119.991455
2,Case 3,0.127971,0.885781,1.480993,198.958098



WORKBOOK: 02 Field Production.xlsx

SHEET: Field Production
Columns: 6
Column names: ['Identifier', 'Date', 'Oil production rate (STB/d)', 'Gas production rate (MSCF/d)', 'Water production rate (STB/d)', 'Water injection rate (STB/d)']


...

,I Index,J Index,K Index,X Coordinate,Y Coordinate,Z Coordinate,Pore Volume (PORV),Porosity (PORO),Permeability I (PERMX),Permeability J (PERMY),Permeability K (PERMZ),Net-to-Gross Ratio (NTG),Transmissibility I (TRANX),Transmissibility J (TRANY),Transmissibility K (TRANZ),Fluid In Place Region (FIPNUM),PVT Region (PVTNUM),Connate Water Saturation (SWL),Critical Water Saturation (SWCR),Maximum Water Saturation (SWU),Connateg Gas Saturation (SGL),Critical Gas Saturation (SGCR),Residual Oil to Water (SOWCR),Pressure (PRESSURE),Water Saturation (SWAT),Oil Saturation (SOIL),Gas Saturation (SGAS),Solution Gas-Oil Ratio (RS),Vapor Oil-Gas Ratio (RV)
0,1,1,31,-999.25000,-9.992500e+02,-999.250000,0.000000,-999.250000,-999.250000,-999.250000,-999.25000,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.2500,-999.25
1,2,1,31,-999.25000,-9.992500e+02,-999.250000,0.000000,-999.250000,-999.250000,-999.250000,-999.25000,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.2500,-999.25
2,3,1,31,550563.24759,6.798611e+06,-8889.945557,20139.761719,0.118908,0.241015,0.241015,0.31234,1.00,0.007884,0.00,0.00,1.00,1.00,0.27,0.35,1.00,0.00,0.10,0.18,4778.917969,1.00,0.00,0.00,0.3633,0.00



WORKBOOK: 08 Test Cases.xlsx

SHEET: Case 86
Columns: 29
Column names: ['I Index', 'J Index', 'K Index', 'X Coordinate', 'Y Coordinate', 'Z Coordinate', 'Pore Volume (PORV)', 'Porosity (PORO)', 'Permeability I (PERMX)', 'Permeability J (PERMY)', 'Permeability K (PERMZ)', 'Net-to-Gross Ratio (NTG)', 'Transmissibility I (TRANX)', 'Transmissibility J (TRANY)', 'Transmissibility K (TRANZ)', 'Fluid In Place Region (FIPNUM)', 'PVT Region (PVTNUM)', 'Connate Water Saturation (SWL)', 'Critical Water Saturation (SWCR)', 'Maximum Water Saturation (SWU)', ' Connateg  Gas Saturation (SGL)', 'Critical Gas Saturation (SGCR)', 'Residual Oil to Water (SOWCR)', 'Pressure (PRESSURE)', 'Water Saturation (SWAT)', 'Oil Saturation (SOIL)', 'Gas Saturation (SGAS)', 'Solution Gas-Oil Ratio (RS)', 'Vapor Oil-Gas Ratio (RV)']


,I Index,J Index,K Index,X Coordinate,Y Coordinate,Z Coordinate,Pore Volume (PORV),Porosity (PORO),Permeability I (PERMX),Permeability J (PERMY),Permeability K (PERMZ),Net-to-Gross Ratio (NTG),Transmissibility I (TRANX),Transmissibility J (TRANY),Transmissibility K (TRANZ),Fluid In Place Region (FIPNUM),PVT Region (PVTNUM),Connate Water Saturation (SWL),Critical Water Saturation (SWCR),Maximum Water Saturation (SWU),Connateg Gas Saturation (SGL),Critical Gas Saturation (SGCR),Residual Oil to Water (SOWCR),Pressure (PRESSURE),Water Saturation (SWAT),Oil Saturation (SOIL),Gas Saturation (SGAS),Solution Gas-Oil Ratio (RS),Vapor Oil-Gas Ratio (RV)
0,1,1,31,-999.25000,-9.992500e+02,-999.250000,0.000000,-999.250000,-999.25000,-999.25000,-999.25000,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.2500,-999.25
1,2,1,31,-999.25000,-9.992500e+02,-999.250000,0.000000,-999.250000,-999.25000,-999.25000,-999.25000,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.25,-999.250000,-999.25,-999.25,-999.25,-999.2500,-999.25
2,3,1,31,550563.24759,6.798611e+06,-8889.945557,14136.852539,0.083466,2.14876,2.14876,0.31234,1.00,0.070292,0.00,0.00,1.00,1.00,0.27,0.35,1.00,0.00,0.10,0.18,4778.917969,1.00,0.00,0.00,0.3633,0.00



WORKBOOK: 09 Template Deliverable.xlsx

SHEET: Case 1
Columns: 4
Column names: ['Date', 'Gas production cumulative [MSCF]', 'Oil production cumulative [STB]', 'Water production cumulative [STB]']


,Date,Gas production cumulative [MSCF],Oil production cumulative [STB],Water production cumulative [STB]
0,04/01/2008,NaN,NaN,NaN
1,07/01/2008,NaN,NaN,NaN
2,10/01/2008,NaN,NaN,NaN


In [19]:


import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

RAW = Path.cwd() / "data" / "raw"

# Sheets we want to inspect
targets = {
    "01 Introduction.xlsx": ["Nomenclature", "Uncertainty Values"],
    "02 Field Production.xlsx": [
        "Field Production",
        "Gas production cumulative",
        "Oil production rate",
        "Oil production cumulative"
    ],
    "03 Train Cases 1.xlsx": ["Case 1"]
}

output = []

for filename, sheets in targets.items():
    # Find the workbook using its filename
    matches = [
        p for p in RAW.glob("*.xlsx")
        if filename.lower().replace(".xlsx", "") in p.stem.lower()
    ]

    if not matches:
        output.append(f"\n### WORKBOOK NOT FOUND: {filename}")
        continue

    file = matches[0]

    for sheet in sheets:
        output.append(f"\n{'=' * 70}")
        output.append(f"WORKBOOK: {file.name} | SHEET: {sheet}")
        output.append("=" * 70)

        try:
            df = pd.read_excel(file, sheet_name=sheet)

            output.append(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            output.append(f"Column names: {df.columns.tolist()}")
            output.append("\nFirst 5 rows:")
            output.append(df.head(5).to_string(index=False))
            output.append("\nLast 3 rows:")
            output.append(df.tail(3).to_string(index=False))

        except Exception as e:
            output.append(f"Could not read sheet: {e}")

# Display everything as ONE output block
result_text = "\n".join(output)

print(result_text)


WORKBOOK: 01 Introduction.xlsx | SHEET: Nomenclature
Shape: 32 rows × 4 columns
Column names: ['Grid', 'I Index', 'Unnamed: 2', 'Unnamed: 3']

First 5 rows:
Grid      I Index Unnamed: 2 Unnamed: 3
 NaN      J Index        NaN        NaN
 NaN      K Index        NaN        NaN
 NaN X Coordinate        NaN        NaN
 NaN Y Coordinate        NaN        NaN
 NaN Z Coordinate        NaN        NaN

Last 3 rows:
Grid I Index             Unnamed: 2                                                    Unnamed: 3
 NaN    SGAS         Gas Saturation The fraction or percentage of the pore volume occupied by gas
 NaN      RS Solution gas-oil ratio               Volume of dissolved gas per unit stock-tank oil
 NaN      RV            Vapor ratio Volume of stock-tank oil associated with a unit volume of gas

WORKBOOK: 01 Introduction.xlsx | SHEET: Uncertainty Values
Shape: 100 rows × 5 columns
Column names: ['Item', 'Fault Transmissibility', 'Porosity Multiplier', 'Permeability Multiplier', 'Aquifer 

Profile Case 1

In [20]:

import pandas as pd
from pathlib import Path

RAW = Path.cwd() / "data" / "raw"
file = next(
    p for p in RAW.glob("*.xlsx")
    if "03 Train Cases 1" in p.stem
)

df = pd.read_excel(file, sheet_name="Case 1")

# Count placeholder values in each column
placeholder_counts = df.eq(-999.25).sum()

# Count actual blank cells in each column
missing_counts = df.isna().sum()

# Summarize the data quality
summary = pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Missing values": missing_counts,
    "Placeholder -999.25": placeholder_counts,
    "Zero values": df.eq(0).sum(),
    "Unique values": df.nunique()
})

print("Case 1 dimensions:", df.shape)
print("\nData quality summary:")
display(summary)

print("\nRows with PORV = 0:",
      (df["Pore Volume (PORV)"] == 0).sum())

print("Rows with at least one -999.25:",
      df.eq(-999.25).any(axis=1).sum())

print("Rows with all 29 values equal to -999.25:",
      df.eq(-999.25).all(axis=1).sum())

Case 1 dimensions: (90365, 29)

Data quality summary:


,Data type,Missing values,Placeholder -999.25,Zero values,Unique values
I Index,int64,0,0,0,53
J Index,int64,0,0,0,55
K Index,int64,0,0,0,31
X Coordinate,float64,0,1395,0,2871
Y Coordinate,float64,0,1395,0,2871
Z Coordinate,float64,0,1395,0,55830
Pore Volume (PORV),float64,0,0,47853,42473
Porosity (PORO),float64,0,47853,0,36786
Permeability I (PERMX),float64,0,47853,0,42070
Permeability J (PERMY),float64,0,47853,0,42070



Rows with PORV = 0: 47853
Rows with at least one -999.25: 47853
Rows with all 29 values equal to -999.25: 0


In [21]:

# Check the relationship between zero pore volume and placeholders

zero_porv = df["Pore Volume (PORV)"] == 0
has_placeholder = df.eq(-999.25).any(axis=1)

print("Total rows:", len(df))

print("\nRows with zero PORV AND a placeholder:",
      (zero_porv & has_placeholder).sum())

print("Rows with zero PORV but NO placeholder:",
      (zero_porv & ~has_placeholder).sum())

print("Rows with positive PORV AND a placeholder:",
      ((df["Pore Volume (PORV)"] > 0) & has_placeholder).sum())

print("Rows with negative PORV:",
      (df["Pore Volume (PORV)"] < 0).sum())

print("\nPORV summary:")
display(df["Pore Volume (PORV)"].describe())

Total rows: 90365

Rows with zero PORV AND a placeholder: 47853
Rows with zero PORV but NO placeholder: 0
Rows with positive PORV AND a placeholder: 0
Rows with negative PORV: 0

PORV summary:


count    9.036500e+04
mean     4.704588e+04
std      2.572445e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.742356e+04
max      2.426922e+06
Name: Pore Volume (PORV), dtype: float64

In [22]:

# Count placeholder values in each column,
# separately for zero-PORV and positive-PORV cells

zero_porv = df["Pore Volume (PORV)"] == 0
positive_porv = df["Pore Volume (PORV)"] > 0

placeholder_check = pd.DataFrame({
    "Placeholder in zero-PORV cells":
        df.loc[zero_porv].eq(-999.25).sum(),
    "Placeholder in positive-PORV cells":
        df.loc[positive_porv].eq(-999.25).sum(),
    "Total placeholder values":
        df.eq(-999.25).sum()
})

display(placeholder_check)

,Placeholder in zero-PORV cells,Placeholder in positive-PORV cells,Total placeholder values
I Index,0,0,0
J Index,0,0,0
K Index,0,0,0
X Coordinate,1395,0,1395
Y Coordinate,1395,0,1395
Z Coordinate,1395,0,1395
Pore Volume (PORV),0,0,0
Porosity (PORO),47853,0,47853
Permeability I (PERMX),47853,0,47853
Permeability J (PERMY),47853,0,47853


In [24]:

# Compare reservoir properties for zero- and positive-PORV cells

zero_porv = df["Pore Volume (PORV)"] == 0
positive_porv = df["Pore Volume (PORV)"] > 0

properties = [
    "Porosity (PORO)",
    "Permeability I (PERMX)",
    "Permeability J (PERMY)",
    "Permeability K (PERMZ)",
    "Pressure (PRESSURE)",
    "Water Saturation (SWAT)",
    "Oil Saturation (SOIL)",
    "Gas Saturation (SGAS)"
]

rows = []

for col in properties:
    rows.append({
        "Property": col,
        "Zero-PORV placeholder count":
            df.loc[zero_porv, col].eq(-999.25).sum(),
        "Positive-PORV placeholder count":
            df.loc[positive_porv, col].eq(-999.25).sum(),
        "Positive-PORV minimum":
            df.loc[positive_porv, col].min(),
        "Positive-PORV maximum":
            df.loc[positive_porv, col].max()
    })

display(pd.DataFrame(rows).round(4))

,Property,Zero-PORV placeholder count,Positive-PORV placeholder count,Positive-PORV minimum,Positive-PORV maximum
0,Porosity (PORO),47853,0,0.0390,0.1991
1,Permeability I (PERMX),47853,0,0.1535,433.0699
2,Permeability J (PERMY),47853,0,0.1535,433.0699
3,Permeability K (PERMZ),47853,0,0.0504,216.5350
4,Pressure (PRESSURE),47853,0,4247.2109,4891.7202
5,Water Saturation (SWAT),47853,0,0.2000,1.0000
6,Oil Saturation (SOIL),47853,0,0.0000,0.8000
7,Gas Saturation (SGAS),47853,0,0.0000,0.0000


Check Saturation Consistency

In [25]:

# Check fluid saturation consistency in positive-PORV cells

positive_cells = df[df["Pore Volume (PORV)"] > 0].copy()

positive_cells["Saturation Sum"] = (
    positive_cells["Water Saturation (SWAT)"]
    + positive_cells["Oil Saturation (SOIL)"]
    + positive_cells["Gas Saturation (SGAS)"]
)

print("Saturation sum statistics:")
display(positive_cells["Saturation Sum"].describe())

print("\nRows where saturation sum differs from 1 by more than 0.01:")
saturation_issues = positive_cells[
    (positive_cells["Saturation Sum"] - 1).abs() > 0.01
]

print("Count:", len(saturation_issues))
display(
    saturation_issues[
        [
            "I Index", "J Index", "K Index",
            "Water Saturation (SWAT)",
            "Oil Saturation (SOIL)",
            "Gas Saturation (SGAS)",
            "Saturation Sum"
        ]
    ].head(10)
)

Saturation sum statistics:


count    42512.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: Saturation Sum, dtype: float64


Rows where saturation sum differs from 1 by more than 0.01:
Count: 0


,I Index,J Index,K Index,Water Saturation (SWAT),Oil Saturation (SOIL),Gas Saturation (SGAS),Saturation Sum


Profile All 70 Training Cases

In [26]:

from pathlib import Path
import pandas as pd

RAW = Path.cwd() / "data" / "raw"
REPORTS = Path.cwd() / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

marker = -999.25
results = []

# Read all four training workbooks
train_files = sorted(
    p for p in RAW.glob("*.xlsx")
    if "train cases" in p.stem.lower()
)

for file in train_files:
    xl = pd.ExcelFile(file)

    for sheet in xl.sheet_names:
        if not sheet.lower().startswith("case"):
            continue

        df_case = pd.read_excel(file, sheet_name=sheet)

        porv = df_case["Pore Volume (PORV)"]
        zero_porv = porv.eq(0)
        positive_porv = porv.gt(0)

        saturation_cols = [
            "Water Saturation (SWAT)",
            "Oil Saturation (SOIL)",
            "Gas Saturation (SGAS)"
        ]

        # Exclude placeholder-containing rows from saturation validation
        valid_saturation = (
            positive_porv
            & ~df_case[saturation_cols].eq(marker).any(axis=1)
            & df_case[saturation_cols].notna().all(axis=1)
        )

        saturation_sum = df_case.loc[
            valid_saturation, saturation_cols
        ].sum(axis=1)

        results.append({
            "workbook": file.name,
            "case": sheet,
            "rows": len(df_case),
            "columns": len(df_case.columns),
            "missing_cells": int(df_case.isna().sum().sum()),
            "placeholder_cells": int(df_case.eq(marker).sum().sum()),
            "rows_with_placeholder": int(
                df_case.eq(marker).any(axis=1).sum()
            ),
            "zero_PORV_rows": int(zero_porv.sum()),
            "positive_PORV_rows": int(positive_porv.sum()),
            "negative_PORV_rows": int(porv.lt(0).sum()),
            "saturation_rows_checked": int(valid_saturation.sum()),
            "saturation_sum_issues": int(
                ((saturation_sum - 1).abs() > 0.01).sum()
            )
        })

report = pd.DataFrame(results)

output_file = REPORTS / "training_cases_quality_profile.csv"
report.to_csv(output_file, index=False)

print("Cases profiled:", len(report))
print("Report saved to:", output_file)
display(report)

Cases profiled: 70
Report saved to: c:\Users\iffah\Downloads\dataworks\reports\training_cases_quality_profile.csv


,workbook,case,rows,columns,missing_cells,placeholder_cells,rows_with_placeholder,zero_PORV_rows,positive_PORV_rows,negative_PORV_rows,saturation_rows_checked,saturation_sum_issues
0,03 Train Cases 1.xlsx,Case 1,90365,29,0,1056951,47853,47853,42512,0,42512,0
1,03 Train Cases 1.xlsx,Case 2,90365,29,0,1056951,47853,47853,42512,0,42512,0
2,03 Train Cases 1.xlsx,Case 3,90365,29,0,1056951,47853,47853,42512,0,42512,0
3,03 Train Cases 1.xlsx,Case 4,90365,29,0,1056951,47853,47853,42512,0,42512,0
4,03 Train Cases 1.xlsx,Case 5,90365,29,0,1056951,47853,47853,42512,0,42512,0
5,03 Train Cases 1.xlsx,Case 6,90365,29,0,1056951,47853,47853,42512,0,42512,0
6,03 Train Cases 1.xlsx,Case 7,90365,29,0,1056951,47853,47853,42512,0,42512,0
7,03 Train Cases 1.xlsx,Case 8,90365,29,0,1056951,47853,47853,42512,0,42512,0
8,03 Train Cases 1.xlsx,Case 9,90365,29,0,1056951,47853,47853,42512,0,42512,0
9,03 Train Cases 1.xlsx,Case 10,90365,29,0,1056951,47853,47853,42512,0,42512,0


Baca nomenclature

In [ ]:

from pathlib import Path
import pandas as pd

RAW = Path.cwd() / "data" / "raw"

intro_file = next(
    p for p in RAW.glob("*.xlsx")
    if "01 Introduction" in p.stem
)

nomenclature = pd.read_excel(
    intro_file,
    sheet_name="Nomenclature",
    header=None
)

print(nomenclature.to_string(index=False, header=False))


In [29]:
import pandas as pd
from pathlib import Path

# Locate the original Case 1 workbook
RAW = Path.cwd() / "data" / "raw"
REPORTS = Path.cwd() / "reports"
PROCESSED = Path.cwd() / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

file = RAW / "03 Train Cases 1.xlsx"

# Load Case 1 without changing the original file
df = pd.read_excel(file, sheet_name="Case 1")

# Make a separate working copy
cleaned_df = df.copy()

print("Original shape:", df.shape)
print("Working copy shape:", cleaned_df.shape)
print("Original file preserved:", file.exists())

Original shape: (90365, 29)
Working copy shape: (90365, 29)
Original file preserved: True


Check Proposed Cleaning Rule 

In [30]:
marker = -999.25

# Identify cells with zero pore volume
zero_pore_volume = cleaned_df["Pore Volume (PORV)"].eq(0)

# Identify reservoir-property columns, excluding coordinates,
# grid indices and pore volume
property_columns = [
    col for col in cleaned_df.columns
    if col not in [
        "I Index",
        "J Index",
        "K Index",
        "X Coordinate",
        "Y Coordinate",
        "Z Coordinate",
        "Pore Volume (PORV)"
    ]
]

# Count placeholder values in zero-PORV cells
placeholder_count = (
    cleaned_df.loc[zero_pore_volume, property_columns]
    .eq(marker)
    .sum()
    .sum()
)

print("Zero-PORV rows:", zero_pore_volume.sum())
print("Placeholder values in their reservoir properties:",
      placeholder_count)

# Verify whether positive-PORV rows contain placeholders
positive_pore_volume = cleaned_df["Pore Volume (PORV)"].gt(0)

positive_placeholder_count = (
    cleaned_df.loc[positive_pore_volume, property_columns]
    .eq(marker)
    .sum()
    .sum()
)

print("Placeholders in positive-PORV rows:",
      positive_placeholder_count)

Zero-PORV rows: 47853
Placeholder values in their reservoir properties: 1052766
Placeholders in positive-PORV rows: 0


Check Coordinate Values
(Some coordinates have -999.25 [unusable])

In [31]:
marker = -999.25

coordinate_columns = [
    "X Coordinate",
    "Y Coordinate",
    "Z Coordinate"
]

# Count placeholder values in each coordinate column
print("Placeholder counts in coordinates:")
print(df[coordinate_columns].eq(marker).sum())

# Count rows where all three coordinates are valid
valid_coordinates = ~df[coordinate_columns].eq(marker).any(axis=1)

print("\nRows with all three valid coordinates:",
      valid_coordinates.sum())

# Compare coordinate validity with pore volume
print("\nCoordinate validity by pore volume:")
print(
    df.assign(Valid_Coordinates=valid_coordinates)
      .groupby("Pore Volume (PORV)")["Valid_Coordinates"]
      .agg(["count", "sum"])
      .head()
)

Placeholder counts in coordinates:
X Coordinate    1395
Y Coordinate    1395
Z Coordinate    1395
dtype: int64

Rows with all three valid coordinates: 88970

Coordinate validity by pore volume:
                    count    sum
Pore Volume (PORV)              
0.000000            47853  46458
0.629475                1      1
3.893268                1      1
5.804942                1      1
5.933069                1      1


Cleaning Summary

In [32]:
# Summarise Case 1 before cleaning

marker = -999.25

summary = pd.DataFrame({
    "Check": [
        "Total rows",
        "Total columns",
        "Zero-PORV rows",
        "Positive-PORV rows",
        "Rows with invalid coordinates",
        "Rows with valid coordinates",
        "Rows with placeholders in property columns"
    ],
    "Count": [
        len(cleaned_df),
        len(cleaned_df.columns),
        cleaned_df["Pore Volume (PORV)"].eq(0).sum(),
        cleaned_df["Pore Volume (PORV)"].gt(0).sum(),
        cleaned_df[coordinate_columns].eq(marker).any(axis=1).sum(),
        (~cleaned_df[coordinate_columns].eq(marker).any(axis=1)).sum(),
        (
            cleaned_df[property_columns].eq(marker)
            .any(axis=1)
            .sum()
        )
    ]
})

display(summary)

,Check,Count
0,Total rows,90365
1,Total columns,29
2,Zero-PORV rows,47853
3,Positive-PORV rows,42512
4,Rows with invalid coordinates,1395
5,Rows with valid coordinates,88970
6,Rows with placeholders in property columns,47853


Apply Cleaning Rule

In [33]:
import numpy as np
from pathlib import Path

# Create a separate copy
cleaned_df = df.copy()

marker = -999.25

# Identify inactive cells
zero_pore_volume = cleaned_df["Pore Volume (PORV)"].eq(0)

# Replace placeholders only in reservoir-property columns
# and only for zero-PORV rows.
cleaning_log = []

for col in property_columns:
    mask = zero_pore_volume & cleaned_df[col].eq(marker)
    changed = int(mask.sum())

    cleaned_df.loc[mask, col] = np.nan

    if changed > 0:
        cleaning_log.append({
            "Column": col,
            "Values converted to missing": changed
        })

cleaning_log_df = pd.DataFrame(cleaning_log)

print("Original shape:", df.shape)
print("Processed shape:", cleaned_df.shape)
print("Placeholder values converted:",
      cleaning_log_df["Values converted to missing"].sum())
print("Remaining placeholders in property columns:",
      cleaned_df[property_columns].eq(marker).sum().sum())

display(cleaning_log_df)

Original shape: (90365, 29)
Processed shape: (90365, 29)
Placeholder values converted: 1052766
Remaining placeholders in property columns: 0


,Column,Values converted to missing
0,Porosity (PORO),47853
1,Permeability I (PERMX),47853
2,Permeability J (PERMY),47853
3,Permeability K (PERMZ),47853
4,Net-to-Gross Ratio (NTG),47853
5,Transmissibility I (TRANX),47853
6,Transmissibility J (TRANY),47853
7,Transmissibility K (TRANZ),47853
8,Fluid In Place Region (FIPNUM),47853
9,PVT Region (PVTNUM),47853


In [34]:
print("Original marker count:",
      df[property_columns].eq(-999.25).sum().sum())

print("Processed missing values in property columns:",
      cleaned_df[property_columns].isna().sum().sum())

print("Original shape unchanged:", df.shape == (90365, 29))
print("Processed shape preserved:", cleaned_df.shape == df.shape)

Original marker count: 1052766
Processed missing values in property columns: 1052766
Original shape unchanged: True
Processed shape preserved: True


In [35]:
# STEP 5: Validate the processed Case 1 data

marker = -999.25

# 1. Check dimensions
print("Original shape:", df.shape)
print("Processed shape:", cleaned_df.shape)

# 2. Check remaining placeholders in property columns
print(
    "Remaining property placeholders:",
    cleaned_df[property_columns].eq(marker).sum().sum()
)

# 3. Check missing values in property columns
print(
    "Missing property values after cleaning:",
    cleaned_df[property_columns].isna().sum().sum()
)

# 4. Check that all grid indices are unchanged
index_columns = ["I Index", "J Index", "K Index"]

print(
    "Grid indices unchanged:",
    df[index_columns].equals(cleaned_df[index_columns])
)

# 5. Check positive-PORV rows remain unchanged
positive_mask = df["Pore Volume (PORV)"].gt(0)

print(
    "Positive-PORV rows unchanged:",
    df.loc[positive_mask].equals(cleaned_df.loc[positive_mask])
)

Original shape: (90365, 29)
Processed shape: (90365, 29)
Remaining property placeholders: 0
Missing property values after cleaning: 1052766
Grid indices unchanged: True
Positive-PORV rows unchanged: True


Clean Case 1

In [36]:
from pathlib import Path

# Create the processed-data folder
PROCESSED = Path.cwd() / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Save Case 1 as a separate Excel file
output_file = PROCESSED / "Case_1_Cleaned.xlsx"

cleaned_df.to_excel(output_file, index=False)

# Confirm that the file exists
print("Saved successfully:", output_file.exists())
print("File location:", output_file)
print("Rows saved:", len(cleaned_df))
print("Columns saved:", len(cleaned_df.columns))

Saved successfully: True
File location: c:\Users\iffah\Downloads\dataworks\data\processed\Case_1_Cleaned.xlsx
Rows saved: 90365
Columns saved: 29


In [37]:
saved_df = pd.read_excel(output_file)

print("Saved file shape:", saved_df.shape)
print(
    "Remaining property placeholders:",
    saved_df[property_columns].eq(-999.25).sum().sum()
)
print(
    "Missing property values:",
    saved_df[property_columns].isna().sum().sum()
)

Saved file shape: (90365, 29)
Remaining property placeholders: 0
Missing property values: 1052766


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path.cwd() / "data" / "raw"
PROCESSED = Path.cwd() / "data" / "processed"
REPORTS = Path.cwd() / "reports"

PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

MARKER = -999.25
case_workbooks = [
    RAW / "03 Train Cases 1.xlsx",
    RAW / "04 Train Cases 2.xlsx",
    RAW / "05 Train Cases 3.xlsx",
    RAW / "06 Train Cases 4.xlsx",
    RAW / "07 Validation Cases.xlsx",
    RAW / "08 Test Cases.xlsx",
]
index_columns = ["I Index", "J Index", "K Index"]
coordinate_columns = ["X Coordinate", "Y Coordinate", "Z Coordinate"]
pore_volume_column = "Pore Volume (PORV)"
summary = []

for input_file in case_workbooks:
    if not input_file.exists():
        print(f"SKIPPED (file not found): {input_file.name}")
        continue

    output_file = PROCESSED / f"{input_file.stem}_Cleaned.xlsx"
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        workbook = pd.ExcelFile(input_file)

        for sheet_name in workbook.sheet_names:
            source_df = pd.read_excel(input_file, sheet_name=sheet_name)

            if not sheet_name.strip().lower().startswith("case "):
                source_df.to_excel(writer, sheet_name=sheet_name, index=False)
                continue

            df = source_df.copy()
            if pore_volume_column not in df.columns:
                raise ValueError(
                    f"{input_file.name} / {sheet_name}: missing {pore_volume_column}"
                )

            property_columns = [
                col for col in df.columns
                if col not in index_columns + coordinate_columns + [pore_volume_column]
            ]
            zero_pv = df[pore_volume_column].eq(0)
            positive_pv = df[pore_volume_column].gt(0)

            marker_before = int(
                df.loc[zero_pv, property_columns].eq(MARKER).sum().sum()
            )
            positive_markers_before = int(
                df.loc[positive_pv, property_columns].eq(MARKER).sum().sum()
            )

            for col in property_columns:
                mask = zero_pv & df[col].eq(MARKER)
                df.loc[mask, col] = np.nan

            remaining_property_markers = int(
                df[property_columns].eq(MARKER).sum().sum()
            )
            positive_rows_unchanged = source_df.loc[positive_pv].equals(
                df.loc[positive_pv]
            )
            indices_unchanged = source_df[index_columns].equals(df[index_columns])
            shape_unchanged = source_df.shape == df.shape
            zero_pv_property_markers_after = int(
                df.loc[zero_pv, property_columns].eq(MARKER).sum().sum()
            )

            if not (positive_rows_unchanged and indices_unchanged and shape_unchanged):
                raise ValueError(f"Validation failed: {input_file.name} / {sheet_name}")
            if zero_pv_property_markers_after != 0:
                raise ValueError(f"Placeholders remain: {input_file.name} / {sheet_name}")

            df.to_excel(writer, sheet_name=sheet_name, index=False)
            summary.append({
                "source_workbook": input_file.name,
                "case": sheet_name,
                "rows": len(source_df),
                "columns": len(source_df.columns),
                "zero_PORV_rows": int(zero_pv.sum()),
                "positive_PORV_rows": int(positive_pv.sum()),
                "property_placeholders_converted": marker_before,
                "positive_PORV_property_placeholders_before": positive_markers_before,
                "remaining_property_placeholders": remaining_property_markers,
                "grid_indices_unchanged": indices_unchanged,
                "positive_PORV_rows_unchanged": positive_rows_unchanged,
                "shape_unchanged": shape_unchanged,
                "validation": "PASS",
            })

    print(f"Saved: {output_file}")

summary_df = pd.DataFrame(summary)
report_file = REPORTS / "case_cleaning_validation_report.csv"
summary_df.to_csv(report_file, index=False)

print("\\n--- BATCH SUMMARY ---")
print("Cases processed:", len(summary_df))
print("Validation passed:", int((summary_df["validation"] == "PASS").sum()))
print("Report saved:", report_file)
if not summary_df.empty:
    display(summary_df.head())
    print("\\nCases per source workbook:")
    display(summary_df.groupby("source_workbook")["case"].count().to_frame("cases_processed"))

DATA QUALITY ASSESSMENT